In [1]:
import os
import numpy as np
from tqdm import tqdm
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

2026-05-07 10:36:14.449544: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778150174.653556      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778150174.712092      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778150175.170058      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778150175.170239      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778150175.170246      57 computation_placer.cc:177] computation placer alr

In [2]:
BASE = "/kaggle/input/datasets/mnhlfuch/vqa-v2/VQA v2/VQA v2"

TRAIN_IMG = BASE + "/Images/train2014"
VAL_IMG   = BASE + "/Images/val2014"
TEST_IMG  = BASE + "/Images/test2015"

# previous notebook output
DATA_DIR = "/kaggle/input/notebooks/urmilvisariya/vqa-project-notebook-1/preprocessed"

train_data = np.load(DATA_DIR + "/train.npy", allow_pickle=True)
val_data   = np.load(DATA_DIR + "/val.npy", allow_pickle=True)
test_data  = np.load(DATA_DIR + "/test.npy", allow_pickle=True)

In [3]:
TRAIN_FEATURE_DIR = "/kaggle/working/features/train"
VAL_FEATURE_DIR   = "/kaggle/working/features/val"
TEST_FEATURE_DIR = "/kaggle/working/features/test"

os.makedirs(TRAIN_FEATURE_DIR, exist_ok=True)
os.makedirs(VAL_FEATURE_DIR, exist_ok=True)
os.makedirs(TEST_FEATURE_DIR, exist_ok=True)

## Loading pre-trained ResNet50 CNN model

base_model.predict(img) will return a feature vector of (batch,7,7,2048)

In [4]:
base_model = ResNet50(weights='imagenet', include_top=False, pooling=None)

I0000 00:00:1778150199.872891      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


<ul>
    <li>rgb image = x*x*3</li>
    <li>resize = 224*224*3</li>
    <li>preproces_input = 224*224*3 -> Normalized pixel values (e.g., -127 to 127) (zero centering)</li>
    <li>expand_dims = 1*224*224*3 where 1 represent batch size</li>
</ul>


In [5]:
def load_image(image_id, folder):
    filename = f"COCO_{folder.split('/')[-1]}_{image_id:012d}.jpg"
    path = os.path.join(folder, filename)
    
    img = Image.open(path).convert("RGB")
    img = img.resize((224, 224))
    
    img = np.array(img)
    img = preprocess_input(img)
    
    return np.expand_dims(img, axis=0)

### Function to extract features from an image

In [6]:
def extract_features(data, img_folder, save_dir, batch_size=64):

    processed = set(os.listdir(save_dir))

    batch_imgs = []
    batch_ids = []

    for item in tqdm(data):

        image_id = item["image_id"]

        file_name = f"{image_id}.npy"

        # skip already processed images
        if file_name in processed:
            continue

        try:

            img = load_image(image_id, img_folder)

            batch_imgs.append(img[0])   # remove batch dim

            batch_ids.append(image_id)

            # process batch
            if len(batch_imgs) == batch_size:

                batch_imgs_np = np.array(batch_imgs)

                feats = base_model.predict(
                    batch_imgs_np,
                    verbose=0
                )

                for feat, img_id in zip(feats, batch_ids):

                    feat = feat.reshape(-1, 2048)

                    feat = feat.astype(np.float16)

                    save_path = os.path.join(
                        save_dir,
                        f"{img_id}.npy"
                    )

                    np.save(save_path, feat)

                batch_imgs = []
                batch_ids = []

        except Exception as e:

            print(f"Error processing {image_id}: {e}")

    # process remaining images
    if len(batch_imgs) > 0:

        batch_imgs_np = np.array(batch_imgs)

        feats = base_model.predict(
            batch_imgs_np,
            verbose=0
        )

        for feat, img_id in zip(feats, batch_ids):

            feat = feat.reshape(-1, 2048)

            feat = feat.astype(np.float16)

            save_path = os.path.join(
                save_dir,
                f"{img_id}.npy"
            )

            np.save(save_path, feat)

#### Extracting image features part by part to avoid storage overloading

### Extracting train image features

In [9]:
extract_features(
    train_data[:150000],
    TRAIN_IMG,
    TRAIN_FEATURE_DIR
)

  0%|          | 59/150000 [00:00<24:04, 103.77it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1778136165.757004     131 service.cc:152] XLA service 0x7a751814a4a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778136165.757043     131 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778136165.757047     131 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778136166.647718     131 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1778136171.610879     131 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
100%|██████████| 150000/150000 [42:51<00:00, 58.33it/s]  


In [10]:
extract_features(
    train_data[150000:300000],
    TRAIN_IMG,
    TRAIN_FEATURE_DIR
)

100%|██████████| 150000/150000 [43:09<00:00, 57.92it/s] 


In [11]:
extract_features(
    train_data[300000:],
    TRAIN_IMG,
    TRAIN_FEATURE_DIR
)

100%|██████████| 143757/143757 [42:15<00:00, 56.70it/s] 


In [14]:
import os

train_files = os.listdir("/kaggle/working/features/train")

print(len(train_files))

82783


### Converting features to zip and uploading all the zip as a dataset for the next notebook input.
#### This notebook cannot store train as well as val image features without overloading output memory

In [21]:
import os
import zipfile
from tqdm import tqdm

train_dir = "/kaggle/working/features/train"

files = os.listdir(train_dir)

chunk_size = 30000

chunk_files = files[:chunk_size]

zip_path = "/kaggle/working/train_chunk1.zip"

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:

    for file in tqdm(chunk_files):

        file_path = os.path.join(train_dir, file)

        zipf.write(
            file_path,
            arcname=file
        )

100%|██████████| 10000/10000 [02:12<00:00, 75.54it/s]


In [22]:
os.remove("/kaggle/working/train_chunk1.zip")

In [23]:
import os
import zipfile
from tqdm import tqdm

train_dir = "/kaggle/working/features/train"

files = os.listdir(train_dir)

chunk_files = files[30000:60000]

zip_path = "/kaggle/working/train_chunk2.zip"

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:

    for file in tqdm(chunk_files):

        file_path = os.path.join(train_dir, file)

        zipf.write(
            file_path,
            arcname=file
        )

100%|██████████| 30000/30000 [05:43<00:00, 87.37it/s]


In [24]:
os.remove("/kaggle/working/train_chunk2.zip")

In [25]:
import os
import zipfile
from tqdm import tqdm

train_dir = "/kaggle/working/features/train"

files = os.listdir(train_dir)

chunk_files = files[60000:]

zip_path = "/kaggle/working/train_chunk3.zip"

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:

    for file in tqdm(chunk_files):

        file_path = os.path.join(train_dir, file)

        zipf.write(
            file_path,
            arcname=file
        )

100%|██████████| 22783/22783 [04:19<00:00, 87.81it/s]


In [ ]:
os.remove("/kaggle/working/train_chunk3.zip")

### Extracting validation data images features

In [7]:
extract_features(
    val_data,
    VAL_IMG,
    VAL_FEATURE_DIR
)

  0%|          | 57/214354 [00:00<36:46, 97.12it/s]  WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1778150210.754579     128 service.cc:152] XLA service 0x79671c14a670 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778150210.754639     128 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1778150211.523109     128 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1778150214.788779     128 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
100%|██████████| 214354/214354 [53:09<00:00, 67.21it/s] 


### Converting validation images features to zip and later downloading it manually

In [11]:
import os
import zipfile
from tqdm import tqdm

val_dir = "/kaggle/working/features/val"

files = os.listdir(val_dir)

zip_path = "/kaggle/working/val_features.zip"

with zipfile.ZipFile(
    zip_path,
    'w',
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for file in tqdm(files):

        file_path = os.path.join(val_dir, file)

        zipf.write(
            file_path,
            arcname=file
        )

100%|██████████| 40504/40504 [07:28<00:00, 90.24it/s]


In [ ]:
os.remove("/kaggle/working/val_features.zip")

### Extracting test data images features

In [8]:
test_data = list(test_data)

In [9]:
import random

random.seed(42)

test_data_subset = random.sample(test_data, 5000)

In [10]:
extract_features(
    test_data_subset,
    TEST_IMG,
    TEST_FEATURE_DIR
)

100%|██████████| 5000/5000 [02:01<00:00, 41.13it/s]


### Converting test data images features to zip

In [12]:
import os
import zipfile
from tqdm import tqdm

test_dir = "/kaggle/working/features/test"

files = os.listdir(test_dir)

zip_path = "/kaggle/working/test_features.zip"

with zipfile.ZipFile(
    zip_path,
    'w',
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for file in tqdm(files):

        file_path = os.path.join(test_dir, file)

        zipf.write(
            file_path,
            arcname=file
        )

100%|██████████| 4750/4750 [00:50<00:00, 94.30it/s] 
